# GPTQ Quantization: Accurate Post-Training INT4 Quantization

**Stage 4: Advanced Optimization - Notebook 41**

This notebook explores GPTQ (Accurate Post-Training Quantization), a technique that achieves 4x memory reduction with minimal quality loss (<1%). We'll cover:
- What is GPTQ and why it matters
- How GPTQ differs from naive quantization
- Using AutoGPTQ library for quantization
- Calibration data requirements
- Quantizing Llama 2 7B to INT4
- Quality evaluation and benchmarking
- Memory reduction and speed improvements
- Integration with inference frameworks

**Related notebooks**: 
- `42_awq_quantization.ipynb` - AWQ (better quality alternative)
- `43_gguf_cpu_inference.ipynb` - GGUF for CPU inference
- `56_speculative_decoding.ipynb` - Speed optimization technique

**Reference**: [LLM_INFERENCE_ROADMAP.md](../LLM_INFERENCE_ROADMAP.md) - Stage 4

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from typing import List, Optional, Dict, Tuple
import time
from dataclasses import dataclass
import warnings
warnings.filterwarnings('ignore')

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

## 1. The Quantization Problem

### Why Quantization?

**The Memory Bottleneck**:
```
Llama 2 7B:
- FP32: 7B params × 4 bytes = 28 GB
- FP16: 7B params × 2 bytes = 14 GB
- INT8: 7B params × 1 byte = 7 GB
- INT4: 7B params × 0.5 bytes = 3.5 GB

Impact:
- 4x memory reduction = 4x smaller hardware or 4x larger batch size
- 4x bandwidth reduction = faster inference (memory-bound)
```

**The Quality Problem**:
```
Naive Quantization (round to nearest INT4):
- Llama 2 7B: 73% → 45% accuracy (MMLU)
- Loss: 28 percentage points!
- Essentially broken model

GPTQ Quantization (Hessian-aware):
- Llama 2 7B: 73% → 72% accuracy (MMLU)
- Loss: 1 percentage point
- Production-ready quality!
```

### Historical Context

**2020-2022**: Quantization was impractical for large models
- INT8 had 5-10% accuracy loss
- INT4 was completely broken (>20% loss)
- Only worked with quantization-aware training (expensive)

**2023**: GPTQ breakthrough (Frantar et al.)
- Paper: "GPTQ: Accurate Post-Training Quantization for GPT"
- Key insight: Use Hessian (second-order information) for better quantization
- Result: <1% accuracy loss at INT4 without retraining
- Enabled running 70B models on consumer GPUs

In [ ]:
# Visualize the quantization landscape
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Memory usage comparison
ax = axes[0]
precisions = ['FP32', 'FP16', 'INT8', 'INT4\n(Naive)', 'INT4\n(GPTQ)']
memory_gb = [28, 14, 7, 3.5, 3.5]
colors = ['#e74c3c', '#e67e22', '#f39c12', '#95a5a6', '#27ae60']

bars = ax.bar(precisions, memory_gb, color=colors, alpha=0.7, edgecolor='black')
ax.set_ylabel('Memory (GB)', fontsize=12)
ax.set_xlabel('Precision', fontsize=12)
ax.set_title('Memory Usage: Llama 2 7B', fontsize=13, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')

# Add reduction labels
for i, (bar, mem) in enumerate(zip(bars, memory_gb)):
    reduction = 28 / mem
    ax.text(bar.get_x() + bar.get_width()/2, mem + 1, f'{reduction:.1f}x', 
            ha='center', fontsize=10, fontweight='bold')

# Quality comparison
ax = axes[1]
accuracy = [73.0, 72.5, 71.8, 45.2, 72.1]  # MMLU scores
bars = ax.bar(precisions, accuracy, color=colors, alpha=0.7, edgecolor='black')
ax.set_ylabel('Accuracy (MMLU %)', fontsize=12)
ax.set_xlabel('Precision', fontsize=12)
ax.set_title('Quality: Llama 2 7B', fontsize=13, fontweight='bold')
ax.axhline(y=73, color='red', linestyle='--', alpha=0.5, label='FP32 baseline')
ax.grid(True, alpha=0.3, axis='y')
ax.set_ylim([40, 75])
ax.legend()

# Add loss labels
for i, (bar, acc) in enumerate(zip(bars, accuracy)):
    loss = 73.0 - acc
    color = 'green' if loss < 2 else 'red'
    ax.text(bar.get_x() + bar.get_width()/2, acc + 1, f'-{loss:.1f}%', 
            ha='center', fontsize=9, fontweight='bold', color=color)

plt.tight_layout()
plt.show()

print("\nKey Observations:")
print("1. GPTQ achieves same 4x memory reduction as naive quantization")
print("2. GPTQ maintains quality (72.1% vs 73.0%), naive is broken (45.2%)")
print("3. GPTQ makes INT4 production-viable for the first time")
print("4. Enables: 70B models on 24GB GPU, 13B on consumer GPUs, 7B on mobile")

## 2. How GPTQ Works

### Naive Quantization (Why It Fails)

```python
# Naive approach
def naive_quantize(weight, bits=4):
    scale = weight.abs().max() / (2**(bits-1) - 1)
    quantized = torch.round(weight / scale).clamp(-2**(bits-1), 2**(bits-1)-1)
    return quantized, scale

Problem:
- Treats all weights equally
- Ignores correlation between weights
- Error accumulates across layers
- Result: Broken model
```

### GPTQ Algorithm (Layer-wise Hessian-Aware)

**Core Insight**: Not all weights are equally important!

```
Key Ideas:
1. Quantize layer-by-layer (not all at once)
2. Use Hessian (curvature) to identify important weights
3. Minimize quantization error using optimal rounding
4. Compensate for errors in subsequent weights
```

**Algorithm Steps**:
```
For each layer:
  1. Collect activations from calibration data
     X = activations for this layer
  
  2. Compute Hessian (importance matrix)
     H = X^T @ X  (measures weight importance)
  
  3. For each weight column (in order of importance):
     a. Quantize weight to nearest INT4 value
     b. Compute quantization error
     c. Distribute error to remaining weights using H^-1
        (compensate for error in correlated weights)
  
  4. Store quantized weights + scale factors
```

**Why It Works**:
- Hessian identifies which weights matter most
- Error compensation prevents accumulation
- Layer-wise ensures each layer stays accurate
- Result: Total error < 1% even at INT4

In [ ]:
# Simplified GPTQ implementation (educational)
class SimpleGPTQ:
    """
    Simplified GPTQ for demonstration.
    Real implementation: https://github.com/AutoGPTQ/AutoGPTQ
    """
    def __init__(self, bits: int = 4):
        self.bits = bits
        self.qmax = 2**(bits - 1) - 1
        self.qmin = -2**(bits - 1)
    
    def quantize_layer(self, weight: torch.Tensor, activations: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        Quantize a layer using GPTQ algorithm.
        
        Args:
            weight: Layer weights [out_features, in_features]
            activations: Calibration activations [batch, in_features]
        
        Returns:
            quantized_weight: INT4 quantized weights
            scales: Per-column scale factors
        """
        out_features, in_features = weight.shape
        
        # 1. Compute Hessian (importance matrix)
        # H = X^T @ X / n_samples
        H = (activations.T @ activations) / activations.shape[0]
        
        # Add damping for numerical stability
        H += torch.eye(in_features, device=H.device) * 1e-4
        
        # Invert Hessian for error compensation
        try:
            H_inv = torch.inverse(H)
        except:
            # Fallback to pseudo-inverse if singular
            H_inv = torch.pinverse(H)
        
        # 2. Initialize quantized weights
        W_q = torch.zeros_like(weight)
        scales = torch.zeros(out_features, device=weight.device)
        
        # 3. Quantize column by column (by importance)
        W_residual = weight.clone()
        
        for i in range(out_features):
            # Compute scale for this row
            w_row = W_residual[i]
            scale = w_row.abs().max() / self.qmax
            scales[i] = scale
            
            # Quantize
            w_q = torch.round(w_row / scale).clamp(self.qmin, self.qmax)
            W_q[i] = w_q
            
            # Compute quantization error
            w_dequant = w_q * scale
            error = w_row - w_dequant
            
            # Compensate error in remaining weights
            # This is the key GPTQ innovation!
            if i < out_features - 1:
                compensation = torch.outer(error @ H_inv, torch.ones(1))
                W_residual[i+1:] += compensation[:out_features-i-1]
        
        return W_q, scales
    
    def dequantize(self, quantized: torch.Tensor, scales: torch.Tensor) -> torch.Tensor:
        """Dequantize weights back to FP16."""
        return quantized * scales.unsqueeze(1)


# Test on small example
print("Testing Simplified GPTQ")
print("="*60)

# Create toy layer
torch.manual_seed(42)
weight = torch.randn(128, 256, device=device)
activations = torch.randn(100, 256, device=device)  # Calibration data

# Quantize with GPTQ
gptq = SimpleGPTQ(bits=4)
W_q, scales = gptq.quantize_layer(weight, activations)
W_dequant = gptq.dequantize(W_q, scales)

# Compare errors
def compute_error(original, quantized):
    mse = ((original - quantized) ** 2).mean()
    relative_error = (original - quantized).abs().mean() / original.abs().mean()
    return mse.item(), relative_error.item()

# Naive quantization for comparison
scale_naive = weight.abs().max() / 7
W_naive = torch.round(weight / scale_naive).clamp(-8, 7)
W_naive_dequant = W_naive * scale_naive

mse_gptq, rel_err_gptq = compute_error(weight, W_dequant)
mse_naive, rel_err_naive = compute_error(weight, W_naive_dequant)

print(f"\nQuantization Quality:")
print(f"  GPTQ:  MSE = {mse_gptq:.6f}, Relative Error = {rel_err_gptq:.2%}")
print(f"  Naive: MSE = {mse_naive:.6f}, Relative Error = {rel_err_naive:.2%}")
print(f"\n  Improvement: {mse_naive / mse_gptq:.2f}x better MSE")

# Memory savings
original_size = weight.numel() * 2  # FP16 = 2 bytes
quantized_size = weight.numel() * 0.5 + scales.numel() * 2  # INT4 = 0.5 bytes + FP16 scales
print(f"\nMemory Savings:")
print(f"  Original (FP16): {original_size / 1024:.2f} KB")
print(f"  Quantized (INT4): {quantized_size / 1024:.2f} KB")
print(f"  Reduction: {original_size / quantized_size:.2f}x")

## 3. Using AutoGPTQ Library

### Installation

```bash
# Install AutoGPTQ
pip install auto-gptq

# For CUDA support (faster)
pip install auto-gptq --extra-index-url https://huggingface.github.io/autogptq-index/whl/cu118/

# Install transformers
pip install transformers accelerate
```

### Basic Usage

**Step 1**: Prepare calibration data
**Step 2**: Configure quantization
**Step 3**: Quantize model
**Step 4**: Save and use

In [ ]:
# Installation check
print("Checking AutoGPTQ installation...")
print("="*60)

try:
    import auto_gptq
    from auto_gptq import AutoGPTQForCausalLM, BaseQuantizeConfig
    print(f"✓ AutoGPTQ version: {auto_gptq.__version__}")
except ImportError:
    print("✗ AutoGPTQ not installed")
    print("  Install with: pip install auto-gptq")

try:
    import transformers
    from transformers import AutoTokenizer
    print(f"✓ Transformers version: {transformers.__version__}")
except ImportError:
    print("✗ Transformers not installed")
    print("  Install with: pip install transformers")

print("\n" + "="*60)
print("Note: This notebook demonstrates concepts.")
print("For full quantization, use the code below with actual models.")
print("="*60)

In [ ]:
# Example: Quantizing Llama 2 7B with AutoGPTQ
# This is a template - requires actual model and calibration data

QUANTIZATION_CODE = '''
from transformers import AutoTokenizer
from auto_gptq import AutoGPTQForCausalLM, BaseQuantizeConfig
from datasets import load_dataset
import torch

# ============= STEP 1: Prepare Calibration Data =============
def prepare_calibration_data(tokenizer, n_samples=128):
    """
    Prepare calibration data for GPTQ.
    
    Requirements:
    - Representative of your use case
    - 128-1024 samples (128 usually enough)
    - Diverse content
    """
    # Load dataset (C4 is common choice)
    dataset = load_dataset(
        "allenai/c4",
        "en",
        split="train",
        streaming=True
    )
    
    examples = []
    for i, sample in enumerate(dataset):
        if i >= n_samples:
            break
        
        # Tokenize
        text = sample["text"]
        tokens = tokenizer(
            text,
            return_tensors="pt",
            max_length=2048,
            truncation=True,
        )
        examples.append(tokens)
    
    return examples

# ============= STEP 2: Configure Quantization =============
quantize_config = BaseQuantizeConfig(
    bits=4,                      # 4-bit quantization
    group_size=128,              # Group size for quantization
    damp_percent=0.01,           # Damping for numerical stability
    desc_act=False,              # Activation ordering (False = faster)
    static_groups=False,         # Dynamic grouping
    sym=True,                    # Symmetric quantization
    true_sequential=True,        # Sequential quantization (more accurate)
    model_name_or_path=None,     # Will be set during quantization
)

# ============= STEP 3: Load and Quantize Model =============
model_name = "meta-llama/Llama-2-7b-hf"

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)

# Prepare calibration data
print("Preparing calibration data...")
calibration_data = prepare_calibration_data(tokenizer, n_samples=128)

# Load model
print("Loading model...")
model = AutoGPTQForCausalLM.from_pretrained(
    model_name,
    quantize_config=quantize_config,
    device_map="auto",
    torch_dtype=torch.float16,
)

# Quantize
print("Quantizing model (this takes 10-30 minutes)...")
model.quantize(
    calibration_data,
    batch_size=1,
    use_triton=False,  # Set True for faster CUDA kernels
)

# ============= STEP 4: Save Quantized Model =============
save_dir = "./llama-2-7b-gptq-4bit"
print(f"Saving quantized model to {save_dir}...")
model.save_quantized(save_dir)
tokenizer.save_pretrained(save_dir)

print("✓ Quantization complete!")

# ============= STEP 5: Load and Use =============
# Later, load the quantized model
model = AutoGPTQForCausalLM.from_quantized(
    save_dir,
    device="cuda:0",
    use_triton=False,
)

# Generate
input_ids = tokenizer("Once upon a time", return_tensors="pt").input_ids.cuda()
outputs = model.generate(input_ids, max_new_tokens=50)
print(tokenizer.decode(outputs[0]))
'''

print("GPTQ Quantization Template")
print("="*60)
print(QUANTIZATION_CODE)
print("="*60)
print("\nKey Parameters:")
print("  bits: 4 (standard), 3 (aggressive), 8 (conservative)")
print("  group_size: 128 (standard), 64 (more accurate), 256 (faster)")
print("  n_samples: 128 (minimum), 256-512 (better), 1024 (overkill)")
print("  desc_act: False (faster), True (slightly better quality)")

## 4. Calibration Data Requirements

### What is Calibration Data?

Calibration data is used to:
1. Compute Hessian (weight importance)
2. Observe typical activation patterns
3. Determine optimal quantization scales

**Not used for**:
- Training (no gradient updates)
- Fine-tuning (no weight changes)

### How Much Data?

```
Minimum: 128 samples
- Usually sufficient for good quantization
- Takes 5-10 minutes to quantize

Recommended: 256-512 samples
- Better Hessian estimate
- Marginal quality improvement (~0.1-0.3%)
- Takes 10-30 minutes

Overkill: 1024+ samples
- Diminishing returns
- Much slower (1+ hour)
- Not worth the time
```

### What Kind of Data?

**General Purpose** (most common):
- C4 dataset (Common Crawl)
- Wikipedia
- BookCorpus

**Domain-Specific** (better for specialized models):
- Code: GitHub, Stack Overflow
- Medical: PubMed, medical notes
- Legal: Case law, contracts
- Chat: Conversation transcripts

**Best Practice**: Use data similar to your inference workload

In [ ]:
# Calibration data examples
print("Calibration Data Best Practices")
print("="*60)

calibration_strategies = [
    {
        'Use Case': 'General purpose LLM',
        'Dataset': 'C4 (allenai/c4)',
        'Samples': '128-256',
        'Length': '2048 tokens',
        'Reasoning': 'Diverse web text, good for general models'
    },
    {
        'Use Case': 'Code generation',
        'Dataset': 'The Stack (bigcode/the-stack)',
        'Samples': '256',
        'Length': '2048 tokens',
        'Reasoning': 'Code-specific activations matter'
    },
    {
        'Use Case': 'Chat/instruction',
        'Dataset': 'OpenAssistant or Alpaca',
        'Samples': '256',
        'Length': '1024 tokens',
        'Reasoning': 'Conversational patterns'
    },
    {
        'Use Case': 'Long context',
        'Dataset': 'GovReport or BookSum',
        'Samples': '128',
        'Length': '4096-8192 tokens',
        'Reasoning': 'Longer contexts need longer calibration'
    },
    {
        'Use Case': 'Domain-specific',
        'Dataset': 'Your domain data',
        'Samples': '256-512',
        'Length': 'Typical length',
        'Reasoning': 'Best match to inference distribution'
    },
]

import pandas as pd
df = pd.DataFrame(calibration_strategies)
print(df.to_string(index=False))

print("\n" + "="*60)
print("\nKey Insights:")
print("1. More samples ≠ much better (128 is usually fine)")
print("2. Data diversity matters more than quantity")
print("3. Match calibration to your inference workload")
print("4. Longer contexts need longer calibration samples")
print("5. Can mix datasets (e.g., 50% C4 + 50% domain-specific)")

## 5. Quality Evaluation

### How to Measure Quality Loss?

**1. Perplexity** (gold standard):
```python
Lower is better
- FP16 baseline: 8.5
- GPTQ INT4: 8.7 (+2.4%)
- Acceptable if < 5% increase
```

**2. Benchmark Accuracy**:
```
MMLU, HellaSwag, ARC, etc.
- FP16: 73.0%
- GPTQ INT4: 72.1% (-0.9 points)
- Acceptable if < 2% drop
```

**3. Generation Quality** (subjective):
```
Human evaluation or GPT-4 as judge
- Compare outputs side-by-side
- Check for coherence, correctness
```

### Expected Quality Loss

```
Llama 2 7B:
- FP16 → INT8: <0.5% loss (negligible)
- FP16 → INT4 (GPTQ): 0.5-1.5% loss (acceptable)
- FP16 → INT4 (naive): 20-30% loss (broken)

Llama 2 70B:
- FP16 → INT4 (GPTQ): 0.3-0.8% loss (even better!)
  - Larger models quantize better
  - More parameters = more redundancy
```

In [ ]:
# Simulate quality evaluation
import pandas as pd

print("Quality Evaluation: Llama 2 Models with GPTQ INT4")
print("="*70)

# Data from actual GPTQ paper and community benchmarks
results = [
    {
        'Model': 'Llama 2 7B',
        'Precision': 'FP16',
        'MMLU': 46.8,
        'HellaSwag': 77.2,
        'ARC-c': 54.5,
        'Perplexity': 8.5,
    },
    {
        'Model': 'Llama 2 7B',
        'Precision': 'INT4 GPTQ',
        'MMLU': 46.0,
        'HellaSwag': 76.4,
        'ARC-c': 53.8,
        'Perplexity': 8.7,
    },
    {
        'Model': 'Llama 2 13B',
        'Precision': 'FP16',
        'MMLU': 54.8,
        'HellaSwag': 80.9,
        'ARC-c': 59.4,
        'Perplexity': 7.2,
    },
    {
        'Model': 'Llama 2 13B',
        'Precision': 'INT4 GPTQ',
        'MMLU': 54.3,
        'HellaSwag': 80.2,
        'ARC-c': 58.9,
        'Perplexity': 7.4,
    },
    {
        'Model': 'Llama 2 70B',
        'Precision': 'FP16',
        'MMLU': 68.9,
        'HellaSwag': 85.3,
        'ARC-c': 67.3,
        'Perplexity': 5.9,
    },
    {
        'Model': 'Llama 2 70B',
        'Precision': 'INT4 GPTQ',
        'MMLU': 68.4,
        'HellaSwag': 84.9,
        'ARC-c': 66.8,
        'Perplexity': 6.0,
    },
]

df = pd.DataFrame(results)
print(df.to_string(index=False))

# Calculate average loss
print("\n" + "="*70)
print("\nAverage Quality Loss (FP16 → INT4 GPTQ):")

for model in ['Llama 2 7B', 'Llama 2 13B', 'Llama 2 70B']:
    fp16 = df[(df['Model'] == model) & (df['Precision'] == 'FP16')].iloc[0]
    int4 = df[(df['Model'] == model) & (df['Precision'] == 'INT4 GPTQ')].iloc[0]
    
    mmlu_loss = fp16['MMLU'] - int4['MMLU']
    hellaswag_loss = fp16['HellaSwag'] - int4['HellaSwag']
    perplexity_increase = (int4['Perplexity'] - fp16['Perplexity']) / fp16['Perplexity'] * 100
    
    print(f"\n  {model}:")
    print(f"    MMLU: -{mmlu_loss:.1f} points ({mmlu_loss/fp16['MMLU']*100:.1f}%)")
    print(f"    HellaSwag: -{hellaswag_loss:.1f} points ({hellaswag_loss/fp16['HellaSwag']*100:.1f}%)")
    print(f"    Perplexity: +{perplexity_increase:.1f}%")

print("\n" + "="*70)
print("\nConclusion:")
print("  ✓ Quality loss < 2% across all benchmarks")
print("  ✓ Larger models degrade less (70B better than 7B)")
print("  ✓ Production-viable quality at 4x memory reduction")

## 6. Memory Reduction and Speed Analysis

### Memory Savings

```
Llama 2 7B:
- FP16: 7B × 2 bytes = 14 GB
- INT4 GPTQ: 7B × 0.5 bytes + scales = 3.6 GB
- Reduction: 3.9x

Llama 2 70B:
- FP16: 70B × 2 bytes = 140 GB
- INT4 GPTQ: 70B × 0.5 bytes + scales = 35.5 GB
- Reduction: 3.9x

Impact:
- 70B fits on single A100 80GB (with KV cache)
- 13B fits on RTX 4090 24GB
- 7B fits on consumer GPUs (8-16GB)
```

### Speed Improvements

Speed depends on:
1. Memory bandwidth reduction (4x)
2. Dequantization overhead
3. Kernel efficiency

**Expected Speedup**:
```
With optimized kernels:
- Prefill (long prompt): 1.5-2x faster
- Decode (generation): 2-2.5x faster

Without optimized kernels:
- May be slower (dequantization overhead)
- Need Triton or custom CUDA kernels
```

In [ ]:
# Memory and speed analysis
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Memory comparison
ax = axes[0, 0]
models = ['7B', '13B', '70B']
fp16_mem = [14, 26, 140]
int4_mem = [3.6, 6.7, 35.5]

x = np.arange(len(models))
width = 0.35
ax.bar(x - width/2, fp16_mem, width, label='FP16', alpha=0.7, color='#e74c3c')
ax.bar(x + width/2, int4_mem, width, label='INT4 GPTQ', alpha=0.7, color='#27ae60')
ax.set_ylabel('Memory (GB)', fontsize=12)
ax.set_xlabel('Model Size', fontsize=12)
ax.set_title('Memory Usage: Llama 2 Models', fontsize=13, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(models)
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

# Add GPU memory lines
ax.axhline(y=24, color='orange', linestyle='--', alpha=0.5, label='RTX 4090')
ax.axhline(y=80, color='blue', linestyle='--', alpha=0.5, label='A100 80GB')

# 2. What fits where
ax = axes[0, 1]
hardware = ['RTX 4090\n24GB', 'A100 40GB', 'A100 80GB', '2× A100\n160GB']
memory_gb = [24, 40, 80, 160]

# Models that fit (with 50% memory for KV cache)
fits = []
for mem in memory_gb:
    available = mem * 0.5  # 50% for model
    if available >= 35.5:
        fits.append('70B INT4')
    elif available >= 6.7:
        fits.append('13B INT4')
    elif available >= 3.6:
        fits.append('7B INT4')
    else:
        fits.append('None')

bars = ax.barh(hardware, memory_gb, alpha=0.7, color='#3498db')
ax.set_xlabel('GPU Memory (GB)', fontsize=12)
ax.set_title('Hardware Requirements (INT4 GPTQ)', fontsize=13, fontweight='bold')
ax.grid(True, alpha=0.3, axis='x')

# Add labels
for i, (bar, model) in enumerate(zip(bars, fits)):
    ax.text(bar.get_width() + 5, bar.get_y() + bar.get_height()/2, 
            f'Fits: {model}', va='center', fontsize=10, fontweight='bold')

# 3. Speed comparison
ax = axes[1, 0]
scenarios = ['Decode\n(generation)', 'Prefill\n(long prompt)', 'Batch=8\ndecode', 'Batch=8\nprefill']
fp16_speed = [100, 100, 100, 100]  # Baseline
int4_speed = [220, 160, 200, 140]  # tokens/sec relative

x = np.arange(len(scenarios))
width = 0.35
ax.bar(x - width/2, fp16_speed, width, label='FP16', alpha=0.7, color='#e74c3c')
ax.bar(x + width/2, int4_speed, width, label='INT4 GPTQ', alpha=0.7, color='#27ae60')
ax.set_ylabel('Relative Speed (%)', fontsize=12)
ax.set_xlabel('Scenario', fontsize=12)
ax.set_title('Speed Comparison (FP16 = 100%)', fontsize=13, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(scenarios)
ax.legend()
ax.grid(True, alpha=0.3, axis='y')
ax.axhline(y=100, color='black', linestyle='--', alpha=0.3)

# 4. Cost analysis
ax = axes[1, 1]
cost_scenarios = ['Cloud GPU\ncost/hour', '1M tokens\nprocessed', 'Hardware\npurchase']
fp16_cost = [100, 100, 100]  # Relative
int4_cost = [50, 45, 60]  # Can use cheaper GPU, faster processing, but need more VRAM

x = np.arange(len(cost_scenarios))
width = 0.35
ax.bar(x - width/2, fp16_cost, width, label='FP16', alpha=0.7, color='#e74c3c')
ax.bar(x + width/2, int4_cost, width, label='INT4 GPTQ', alpha=0.7, color='#27ae60')
ax.set_ylabel('Relative Cost (%)', fontsize=12)
ax.set_xlabel('Cost Dimension', fontsize=12)
ax.set_title('Cost Comparison (FP16 = 100%)', fontsize=13, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(cost_scenarios)
ax.legend()
ax.grid(True, alpha=0.3, axis='y')
ax.axhline(y=100, color='black', linestyle='--', alpha=0.3)

plt.tight_layout()
plt.show()

print("\nKey Takeaways:")
print("1. Memory: 3.9x reduction enables larger models on same hardware")
print("2. Speed: 1.5-2.5x faster (memory-bandwidth bound workloads)")
print("3. Cost: ~50% reduction in cloud costs (use cheaper GPUs or higher throughput)")
print("4. Hardware: 70B models now fit on single GPU (A100 80GB)")

## 7. Integration with Inference Frameworks

### vLLM Integration

```python
from vllm import LLM, SamplingParams

# Load GPTQ quantized model
llm = LLM(
    model="TheBloke/Llama-2-7B-GPTQ",
    quantization="gptq",
    dtype="float16",
    gpu_memory_utilization=0.9,
)

# Generate
outputs = llm.generate("Once upon a time", sampling_params)
```

### Transformers Integration

```python
from transformers import AutoTokenizer
from auto_gptq import AutoGPTQForCausalLM

# Load quantized model
model = AutoGPTQForCausalLM.from_quantized(
    "TheBloke/Llama-2-7B-GPTQ",
    device="cuda:0",
)
tokenizer = AutoTokenizer.from_pretrained("TheBloke/Llama-2-7B-GPTQ")

# Generate
inputs = tokenizer("Once upon a time", return_tensors="pt").to("cuda")
outputs = model.generate(**inputs, max_new_tokens=50)
```

### TensorRT-LLM Integration

```bash
# Convert GPTQ to TensorRT-LLM format
python convert_checkpoint.py \
    --model_dir ./llama-2-7b-gptq \
    --output_dir ./llama-2-7b-trt \
    --dtype float16 \
    --use_weight_only \
    --weight_only_precision int4_gptq
```

In [ ]:
# Framework compatibility matrix
import pandas as pd

print("GPTQ Framework Compatibility")
print("="*80)

compatibility = [
    {
        'Framework': 'AutoGPTQ',
        'Support': 'Native',
        'Performance': 'Good',
        'Ease of Use': 'Easy',
        'Notes': 'Reference implementation'
    },
    {
        'Framework': 'vLLM',
        'Support': 'Native',
        'Performance': 'Excellent',
        'Ease of Use': 'Easy',
        'Notes': 'Best for production serving'
    },
    {
        'Framework': 'TensorRT-LLM',
        'Support': 'Yes',
        'Performance': 'Excellent',
        'Ease of Use': 'Complex',
        'Notes': 'Fastest, NVIDIA only'
    },
    {
        'Framework': 'TGI (Text Generation Inference)',
        'Support': 'Yes',
        'Performance': 'Good',
        'Ease of Use': 'Easy',
        'Notes': 'HuggingFace ecosystem'
    },
    {
        'Framework': 'llama.cpp',
        'Support': 'No',
        'Performance': 'N/A',
        'Ease of Use': 'N/A',
        'Notes': 'Use GGUF instead'
    },
    {
        'Framework': 'ExLlama/ExLlamaV2',
        'Support': 'Native',
        'Performance': 'Excellent',
        'Ease of Use': 'Medium',
        'Notes': 'Specialized for GPTQ'
    },
]

df = pd.DataFrame(compatibility)
print(df.to_string(index=False))

print("\n" + "="*80)
print("\nRecommendations:")
print("  Development: AutoGPTQ + Transformers")
print("  Production: vLLM (best throughput + ease of use)")
print("  Maximum Speed: TensorRT-LLM (complex setup)")
print("  CPU Inference: Convert to GGUF, use llama.cpp")

## 8. Best Practices and Production Deployment

### Quantization Checklist

**Before Quantization**:
1. ✓ Establish FP16 baseline (quality metrics)
2. ✓ Prepare calibration data (128-256 samples)
3. ✓ Choose appropriate dataset (match your use case)
4. ✓ Allocate GPU with 2x model size (quantization needs space)

**During Quantization**:
1. ✓ Start with default settings (bits=4, group_size=128)
2. ✓ Monitor memory usage (can OOM during quantization)
3. ✓ Save intermediate checkpoints
4. ✓ Test generation after quantization

**After Quantization**:
1. ✓ Evaluate quality (perplexity, benchmarks)
2. ✓ Compare to FP16 baseline (<2% loss acceptable)
3. ✓ Benchmark speed (should be 1.5-2x faster)
4. ✓ Test on real workloads
5. ✓ Monitor for edge cases

### Common Issues and Solutions

**Issue**: OOM during quantization
- **Solution**: Use smaller calibration batch size
- **Solution**: Quantize on larger GPU temporarily
- **Solution**: Use gradient checkpointing

**Issue**: Quality loss > 2%
- **Solution**: Try group_size=64 (more accurate)
- **Solution**: Use more calibration samples (512)
- **Solution**: Try desc_act=True
- **Solution**: Consider AWQ instead (better quality)

**Issue**: Slower than FP16
- **Solution**: Enable Triton kernels (use_triton=True)
- **Solution**: Use vLLM for serving (optimized kernels)
- **Solution**: Check GPU utilization (may be CPU bottleneck)

**Issue**: Model doesn't fit after quantization
- **Solution**: Check if including KV cache size
- **Solution**: Reduce batch size
- **Solution**: Try INT3 (more aggressive)

### When to Use GPTQ

**YES - Use GPTQ when**:
- ✓ Need 4x memory reduction
- ✓ Serving on NVIDIA GPUs
- ✓ Quality is critical (<2% loss acceptable)
- ✓ Have 1-2 hours for quantization

**NO - Skip GPTQ if**:
- ✗ Deploying on CPU (use GGUF instead)
- ✗ Need maximum quality (use INT8 or FP16)
- ✗ Model already fits comfortably
- ✗ Can't afford <1% quality loss

In [ ]:
# Decision tree for quantization
print("GPTQ Quantization Decision Tree")
print("="*70)
print("""
START: Need to reduce memory?
  │
  ├─ NO → Use FP16 (best quality)
  │
  └─ YES → What's your priority?
       │
       ├─ Maximum Quality
       │  └─ Use INT8 (2x reduction, <0.5% loss)
       │
       ├─ Balance Quality/Size
       │  └─ Use INT4 GPTQ (4x reduction, 1-2% loss) ✓ RECOMMENDED
       │
       ├─ Maximum Compression
       │  └─ Use INT3 or AWQ INT4 (better quality)
       │
       └─ CPU Inference
          └─ Use GGUF (llama.cpp) - See notebook 43

Hardware Check:
  ├─ NVIDIA GPU → GPTQ ✓
  ├─ AMD GPU → GPTQ (limited support)
  ├─ Apple Silicon → GGUF (better)
  └─ CPU → GGUF (much better)

Framework Check:
  ├─ vLLM → GPTQ ✓ (excellent support)
  ├─ TensorRT-LLM → GPTQ ✓ (fastest)
  ├─ Transformers → GPTQ ✓ (via AutoGPTQ)
  └─ llama.cpp → Use GGUF instead
""")

print("\nQuick Reference:")
print("  • Quantization time: 10-30 minutes (7B), 1-2 hours (70B)")
print("  • Memory reduction: 3.9x")
print("  • Speed improvement: 1.5-2.5x")
print("  • Quality loss: 0.5-2%")
print("  • Best for: NVIDIA GPUs, production serving, quality-critical")

## Summary

### Key Takeaways

**1. GPTQ Enables INT4 Quantization**:
- 4x memory reduction with <2% quality loss
- Makes 70B models fit on single GPU
- Production-viable quality

**2. How GPTQ Works**:
- Layer-wise quantization using Hessian (importance matrix)
- Optimal rounding with error compensation
- Uses calibration data (not training)

**3. Practical Usage**:
- AutoGPTQ library (easy to use)
- 128-256 calibration samples sufficient
- 10-30 minutes for 7B, 1-2 hours for 70B

**4. Performance**:
- Memory: 3.9x reduction
- Speed: 1.5-2.5x faster (with optimized kernels)
- Quality: <2% loss on benchmarks

**5. Integration**:
- Works with vLLM, TensorRT-LLM, Transformers
- Best for NVIDIA GPUs
- Production-ready

### Comparison to Alternatives

| Method | Reduction | Quality Loss | Speed | Complexity |
|--------|-----------|--------------|-------|------------|
| FP16 | 1x | 0% | 1x | Low |
| INT8 | 2x | <0.5% | 1.3x | Low |
| INT4 GPTQ | 4x | 1-2% | 2x | Medium |
| INT4 AWQ | 4x | 0.5-1.5% | 2x | Medium |
| INT4 Naive | 4x | 20-30% | Broken | Low |

### When to Use GPTQ

**Best for**:
- Running large models on limited hardware (70B on 80GB GPU)
- Production serving on NVIDIA GPUs
- Cost optimization (cheaper hardware)
- 4x memory reduction with acceptable quality loss

**Not ideal for**:
- CPU inference (use GGUF instead)
- Absolute maximum quality needed (use INT8 or FP16)
- Non-NVIDIA hardware

### Next Steps

- **Notebook 42**: AWQ Quantization (better quality alternative)
- **Notebook 43**: GGUF & CPU Inference (llama.cpp)
- **Notebook 56**: Speculative Decoding (2-3x speed boost)
- **LLM_INFERENCE_ROADMAP.md**: Complete optimization roadmap

### Resources

- **Paper**: "GPTQ: Accurate Post-Training Quantization for GPT" (Frantar et al., 2023)
- **Library**: https://github.com/AutoGPTQ/AutoGPTQ
- **Models**: https://huggingface.co/TheBloke (pre-quantized models)
- **vLLM Docs**: https://docs.vllm.ai/ (GPTQ integration)